In [7]:
import pandas as pd
import numpy as np

In [32]:
df = pd.read_parquet('../data/master/index_meta.parquet')
df.columns

Index(['date', 'scode', 'is_candidate_tac', 'is_candidate_str',
       'log_market_cap', 'Future_High_Tac', 'Future_Low_Tac',
       'Future_Close_Tac', 'Future_High_Str', 'Future_Low_Str',
       'Future_Close_Str', 'target_ret_5', 'target_tac_vol_scaled_asym_return',
       'target_tac_vol_scaled_asym_return_clipped', 'target_tac_max_neg_path',
       'target_tac_risk', 'target_meta_survival_return_raw', 'target_ret_60',
       'target_str_sharpe_adj', 'target_str_triple_barrier', 'target_str_mdd',
       'target_str_vol_scaled_mdd', 'target_tac_rank', 'target_tac_gauss_rank',
       'target_tac_linear_residual', 'target_tac_sector_relative',
       'target_str_rank', 'target_str_gauss_rank', 'target_str_peer_alpha',
       'target_tac_risk_sev', 'target_tac_risk_5', 'target_tac_risk_7',
       'target_tac_risk_10', 'target_tac_risk_class', 'target_tac_alpha_class',
       'target_tac_alpha_upclass', 'target_tac_alpha_qlclass',
       'target_tac_tb_5_2', 'target_tac_tb_7_2', 'target

In [28]:
df[['Future_High_Tac', 'Future_Low_Tac', 'Future_Close_Tac']].describe()

,Future_High_Tac,Future_Low_Tac,Future_Close_Tac
count,6.645288e+06,6.645288e+06,6.631772e+06
mean,1.038398e+00,9.670703e-01,1.002793e+00
std,1.322624e+00,4.245310e-02,1.244337e+00
min,1.000000e+00,9.938730e-03,1.036463e-02
25%,1.009073e+00,9.564767e-01,9.772727e-01
50%,1.022315e+00,9.777610e-01,1.000000e+00
75%,1.045359e+00,9.904762e-01,1.022785e+00
max,1.701000e+03,1.000000e+00,1.692000e+03


### Triple Barrier

In [33]:
for tp,sl in [(5,2), (7,2), (10,2), (5,3), (7,3), (10,3), (7,4), (10,4)]:
    mfe_5 = df["Future_High_Tac"]
    mae_5 = df["Future_Low_Tac"]
    th_up = 1+tp/100
    th_down = 1-sl/100
    df[f"target_tac_tb_{tp}_{sl}"] = (
        (mfe_5 >= th_up) 
        & (mae_5 >= th_down)
        & (df["Future_Close_Tac"] >= 1.03)
    ).astype("float32")

In [30]:
tp = 10
sl = 2
counts_by_month = (
    df[df['is_candidate_tac']]
    .groupby(df.loc[df['is_candidate_tac'], 'date'].dt.to_period('M'))[f"target_tac_tb_{tp}_{sl}"]
    .value_counts()
    .unstack(fill_value=0)
)
counts_by_month.to_csv('target_tac_tb_counts_by_month.csv')
counts_by_month

target_tac_tb_10_2,0.0,1.0
date,,
2017-01,5742,93
2017-02,6099,138
2017-03,6730,69
2017-04,6115,195
2017-05,6603,186
...,...,...
2026-01,6594,361
2026-02,6695,581
2026-03,9444,386


In [34]:
df.to_parquet('../data/master/index_meta.parquet')

### Others

In [4]:
# target_tac_alpha_class : Closeベースでの分類
upside_severity = [np.log(x) for x in df['Future_Close_Tac']]
th_5  = np.log(1.05)
th_7  = np.log(1.07)
th_10 = np.log(1.10)
cls = np.zeros(len(df), dtype="float32")
cls[upside_severity >= th_5] = 1
cls[upside_severity >= th_7] = 2
cls[upside_severity >= th_10] = 3
df["target_tac_alpha_class"] = cls
df['target_tac_alpha_class'].value_counts()

target_tac_alpha_class
0.0    5921870
1.0     296668
3.0     219804
2.0     206946
Name: count, dtype: int64

In [5]:
# target_tac_alpha_upclass : Highベースでの分類
th_5  = np.log(1.05)
th_7  = np.log(1.07)
th_10 = np.log(1.10)
# 高値到達だけでなく、5日後Closeも極端に弱くないことを要求
r_close_5 = np.log(df['Future_Close_Tac'])
r_high_5 = np.log(df['Future_High_Tac'])
r_low_5  = np.log(df['Future_Low_Tac'])
alpha_event_5 = (r_high_5 >= th_5)  & (r_close_5 > 0.0)
alpha_event_7 = (r_high_5 >= th_7)  & (r_close_5 > 0.0)
alpha_event_10 = (r_high_5 >= th_10) & (r_close_5 > 0.0)
cls = np.zeros(len(df), dtype="float32")
cls[alpha_event_5] = 1
cls[alpha_event_7] = 2
cls[alpha_event_10] = 3
df["target_tac_alpha_upclass"] = cls
df['target_tac_alpha_upclass'].value_counts()

target_tac_alpha_upclass
0.0    5381581
1.0     479498
3.0     427602
2.0     356607
Name: count, dtype: int64

In [5]:
# target_tac_alpha_qlclass : Highベースでの分類 + 5日後の値動きの質も考慮
r_close_5 = np.log(df['Future_Close_Tac'])
r_high_5 = np.log(df['Future_High_Tac'])
r_low_5  = np.log(df['Future_Low_Tac'])
mfe_5 = np.maximum(r_high_5, 0.0)      # maximum favorable excursion
mae_5 = np.maximum(-r_low_5, 0.0)      # maximum adverse excursion
close_pos = r_close_5
upside_quality = (
    0.50 * r_close_5
  + 0.25 * mfe_5
  - 0.25 * mae_5
)
th_5  = np.log(1.05)
th_7  = np.log(1.07)
th_10 = np.log(1.10)
cls = np.zeros(len(df), dtype="float32")
cls[upside_quality >= th_5] = 1
cls[upside_quality >= th_7] = 2
cls[upside_quality >= th_10] = 3
df["target_tac_alpha_qlclass"] = cls
df['target_tac_alpha_qlclass'].value_counts()

target_tac_alpha_qlclass
0.0    6148193
1.0     224896
2.0     140671
3.0     131528
Name: count, dtype: int64

In [6]:
df.to_parquet('../data/master/index_meta.parquet')